In [7]:
!pip install langchain-core -q
!pip install langchain -q

In [ ]:
pip install pypdf


In [ ]:
!pip install langchain langchain-community langchain-openai langchain-groq langchain-text-splitters langchain-core faiss-cpu pypdf2 sentence-transformers streamlit pyngrok -q

In [27]:
!pip install langchain -q
!pip install langchain-groq -q

In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load both reports
loader1 = PyPDFLoader("/content/RIL-Integrated-Annual-Report-2023-24.pdf")
loader2 = PyPDFLoader("/content/RIL-Integrated-Annual-Report-2022-23.pdf")

docs1 = loader1.load()
docs2 = loader2.load()


for doc in docs1:
    doc.metadata["source"] = "RIL Annual Report 2023-24"
for doc in docs2:
    doc.metadata["source"] = "RIL Annual Report 2022-23"

all_docs = docs1 + docs2

print(f"Total pages loaded: {len(all_docs)}")
print(f"Sample metadata: {all_docs[0].metadata}")

Total pages loaded: 426
Sample metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Windows)', 'creationdate': '2024-08-07T11:55:42+05:30', 'moddate': '2024-08-07T11:59:55+05:30', 'trapped': '/False', 'source': 'RIL Annual Report 2023-24', 'total_pages': 159, 'page': 0, 'page_label': '1'}


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = splitter.split_documents(all_docs)

print(f"Total pages loaded: {len(all_docs)}")
print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[10].page_content[:300]}")
print(f"\nChunk source: {chunks[10].metadata['source']}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading embedding model... ")


embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

print("Building FAISS vector store...")

vectorstore = FAISS.from_documents(chunks, embeddings)


vectorstore.save_local("/content/ril_vectorstore")

print("Vector store built and saved!")
print(f"Total vectors indexed: {vectorstore.index.ntotal}")

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os


# LLM
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key="YOUR-KEY",
    temperature=0
)

# Prompt template
prompt_template = PromptTemplate.from_template("""
You are a financial analyst assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say "I couldn't find this in the provided reports."
Always mention which report year your answer is from.

Context: {context}

Question: {question}

Answer:
""")

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Helper to format retrieved docs
def format_docs(docs):
    return "\n\n".join([
        f"[{doc.metadata['source']}, Page {doc.metadata.get('page','N/A')}]\n{doc.page_content}"
        for doc in docs
    ])

# Modern RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")

# Test question
query = "What was Reliance's total revenue in FY24?"
answer = rag_chain.invoke(query)
print(f"\nQuestion: {query}")
print(f"\nAnswer: {answer}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.load_local(
    "/content/ril_vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Vector store loaded! Total vectors: {vectorstore.index.ntotal}")

In [ ]:
import os

# Create the folder structure FAISS expects
os.makedirs("/content/ril_vectorstore", exist_ok=True)

# Move both files into the folder
import shutil
shutil.move("/content/index.faiss", "/content/ril_vectorstore/index.faiss")
shutil.move("/content/index.pkl", "/content/ril_vectorstore/index.pkl")

print("Files organized!")

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# LLM
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key="your-groq-key-here",
    temperature=0
)

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

# Prompt with memory
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a financial analyst assistant for Reliance Industries.
Use ONLY the context below to answer questions.
If the answer is not in the context, say 'I couldn't find this in the reports.'
Always mention which report year your answer is from.

Context: {context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# Chat history storage
chat_history = []

def format_docs(docs):
    return "\n\n".join([
        f"[{doc.metadata['source']}, Page {doc.metadata.get('page','N/A')}]\n{doc.page_content}"
        for doc in docs
    ])

def chat(question):
    # Retrieve relevant docs
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # Build chain with history
    chain = prompt | llm

    response = chain.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })

    # Save to history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))

    return response.content

print("Conversational RAG ready!")

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os



llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key="YOUR-KEY-HERE",
    temperature=0
)


prompt_template = PromptTemplate.from_template("""
You are a financial analyst assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say "I couldn't find this in the provided reports."
Always mention which report year your answer is from.

Context: {context}

Question: {question}

Answer:
""")


retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


def format_docs(docs):
    return "\n\n".join([
        f"[{doc.metadata['source']}, Page {doc.metadata.get('page','N/A')}]\n{doc.page_content}"
        for doc in docs
    ])


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")

#Test
query = "What was Reliance's total revenue in FY24?"
answer = rag_chain.invoke(query)
print(f"\nQuestion: {query}")
print(f"\nAnswer: {answer}")

In [ ]:

chat_history = []
print("History cleared, length:", len(chat_history))

In [ ]:
!pip install streamlit pyngrok -q

In [36]:
%%writefile app.py

import streamlit as st
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Page config
st.set_page_config(
    page_title="RIL Financial Analyst",
    page_icon="📈",
    layout="centered"
)

st.title("📈 Reliance Industries Financial Analyst")
st.caption("Ask questions about RIL Annual Reports 2022-23 and 2023-24")

# Load vector store
@st.cache_resource
def load_vectorstore():
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"}
    )
    vectorstore = FAISS.load_local(
        "/content/ril_vectorstore",
        embeddings,
        allow_dangerous_deserialization=True
    )
    return vectorstore

# Load LLM
@st.cache_resource
def load_llm():
    return ChatGroq(
        model_name="llama-3.1-8b-instant",
        api_key="YOUT-KEY",
        temperature=0
    )

vectorstore = load_vectorstore()
llm = load_llm()
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a financial analyst assistant for Reliance Industries.
Use ONLY the context below to answer questions.
If the answer is not in the context, say 'I couldn't find this in the reports.'
Always mention which report year your answer is from.

Context: {context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join([
        f"[{doc.metadata['source']}, Page {doc.metadata.get('page','N/A')}]\n{doc.page_content}"
        for doc in docs
    ])

# Chat history
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Chat input
if question := st.chat_input("Ask about Reliance Industries..."):
    # Show user message
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    # Get answer
    with st.chat_message("assistant"):
        with st.spinner("Searching reports..."):
            docs = retriever.invoke(question)
            context = format_docs(docs)

            chain = prompt | llm
            response = chain.invoke({
                "context": context,
                "chat_history": st.session_state.chat_history,
                "question": question
            })

            answer = response.content
            st.markdown(answer)

            # Show sources
            with st.expander("📄 Sources"):
                for doc in docs[:3]:
                    st.caption(f"• {doc.metadata['source']}, Page {doc.metadata.get('page','N/A')}")

    # Save to history
    st.session_state.messages.append({"role": "assistant", "content": answer})
    st.session_state.chat_history.append(HumanMessage(content=question))
    st.session_state.chat_history.append(AIMessage(content=answer))

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import subprocess
import time
ngrok.set_auth_token("YOUR-KEY")
# Start streamlit
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501)
print(f"\n App is live at: {public_url}")
